Connected to Python 3.13.0

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse

# Set up R environment
os.environ["R_HOME"] = "/Library/Frameworks/R.framework/Resources"
r_path = "/Library/Frameworks/R.framework/Resources/bin"
os.environ["PATH"] = r_path + ":" + os.environ["PATH"]

# Import rpy2 for R magic
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter
import rpy2.robjects.packages as rpackages

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "PWD" redefined by R and overriding existing variable. Current: "/", R: "/Users/valentingoupille/Internship_Val_Goupille_2025/code/Local_analysis"
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "XPC_SERVICE_NAME" redefined by R and overriding existing variable. Current: "application.com.todesktop.230313mzl4w4u92.106351087.106351093", R: "0"
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_SESSION_TMPDIR" redefined by R and overriding existing variable. Current: "/var/folders/m_/d2crsk_n0mxcl8sqs3mqkwt80000gn/T//RtmpQfgwhv", R: "/var/folders/m_/d2crsk_n0mxcl8sqs3mqkwt80000gn/T//RtmpJlTX8b"
  warnings.warn(


In [ ]:
# Set working directory
os.chdir(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))

In [ ]:
# Load the filtered data from 3c
print("📂 Loading filtered data from 3c...")
INPUT_PATH = "data/Anndata/filtered"
INPUT_FILE = "GeneFull_raw_unique_mult_with_metadata_grouped_filtered_100UMI_3c.h5ad"

# Load the AnnData object
adata_final = ad.read_h5ad(os.path.join(INPUT_PATH, INPUT_FILE))

print(f"📊 Loaded dataset: {adata_final.n_obs:,} cells × {adata_final.n_vars:,} genes")
print(f"🔢 Available obs columns: {list(adata_final.obs.columns)}")
print(f"🔢 Available var columns: {list(adata_final.var.columns)}")

📂 Loading filtered data from 3c...
📊 Loaded dataset: 2,988 cells × 5,500 genes
🔢 Available obs columns: ['barcode', 'Well_loc_rd1', 'Well_loc_rd2', 'Well_loc_rd3', 'CultureMedium', 'RepBio', 'OD', 'RepTech', 'ODt', 'OD_theoric', 'OD_measured', 'Extracted_Sequence_rd1', 'Group', 'n_genes', 'RepTech_suffix', 'RepBio_suffix', 'group', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes']
🔢 Available var columns: ['gene_id', 'final_gene_name', 'symbol_renamed', 'Gene Type', 'Name', 'Protein accession', 'symbol_old', 'locus_tag_old', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'gene_pass_union_OD_5pct', 'gene_pass_intersection_OD_5pct', 'gene_pass_union_OD_10pct', 'gene_pass_intersection_OD_10pct', 'gene_pass_union_OD_15pct', 'gene_pass_intersection_OD_

In [ ]:
# Install and load required R packages
print("\n📦 Setting up R packages...")

ro.r(
    """
# Function to install package if not already installed
install_if_missing <- function(package_name) {
    if (!require(package_name, character.only = TRUE, quietly = TRUE)) {
        cat("Installing", package_name, "...\n")
        if (package_name %in% c("Seurat", "SeuratObject")) {
            install.packages(package_name, repos = "https://cloud.r-project.org/")
        } else if (package_name == "anndata") {
            install.packages("reticulate", repos = "https://cloud.r-project.org/")
            reticulate::py_install("anndata")
        }
        library(package_name, character.only = TRUE)
    }
}

# Install required packages
install_if_missing("Seurat")
install_if_missing("SeuratObject")
install_if_missing("Matrix")
install_if_missing("reticulate")

# Load packages
library(Seurat)
library(SeuratObject)  
library(Matrix)
library(reticulate)

cat("✅ R packages loaded successfully\n")
"""
)


📦 Setting up R packages...


R callback write-console: 
Attachement du package : ‘SeuratObject’

  
R callback write-console: Les objets suivants sont masqués depuis ‘package:base’:

    intersect, t

  



    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    ✅ R packages loaded successfully


In [ ]:
# Convert AnnData to Seurat using R magic
print("\n🔄 Converting AnnData to Seurat V5...")

# Prepare data for R
print("📋 Preparing data for R conversion...")

# Get counts matrix (make sure it's in the right format)
if scipy.sparse.issparse(adata_final.X):
    counts_matrix = adata_final.X.toarray()
else:
    counts_matrix = adata_final.X

# Ensure counts are integers
counts_matrix = counts_matrix.astype(int)

# Get cell metadata
cell_metadata = adata_final.obs.copy()
gene_names = adata_final.var_names.tolist()
cell_names = adata_final.obs_names.tolist()

# Get gene metadata
gene_metadata = adata_final.var.copy()

print(f"📊 Matrix shape: {counts_matrix.shape}")
print(f"📊 Cell metadata shape: {cell_metadata.shape}")
print(f"📊 Gene metadata shape: {gene_metadata.shape}")
print(f"📊 Gene metadata columns: {list(gene_metadata.columns)}")


🔄 Converting AnnData to Seurat V5...
📋 Preparing data for R conversion...
📊 Matrix shape: (2988, 5500)
📊 Cell metadata shape: (2988, 25)
📊 Gene metadata shape: (5500, 21)
📊 Gene metadata columns: ['gene_id', 'final_gene_name', 'symbol_renamed', 'Gene Type', 'Name', 'Protein accession', 'symbol_old', 'locus_tag_old', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'gene_pass_union_OD_5pct', 'gene_pass_intersection_OD_5pct', 'gene_pass_union_OD_10pct', 'gene_pass_intersection_OD_10pct', 'gene_pass_union_OD_15pct', 'gene_pass_intersection_OD_15pct']


In [ ]:
# Convert to R and create Seurat object
print("🔄 Creating Seurat object in R...")

# Transfer data to R using the new converter approach
with localconverter(ro.default_converter + pandas2ri.converter):
    r_cell_metadata = ro.conversion.py2rpy(cell_metadata)
    r_gene_metadata = ro.conversion.py2rpy(gene_metadata)

with localconverter(ro.default_converter + numpy2ri.converter):
    r_counts = ro.conversion.py2rpy(counts_matrix.T)  # Transpose for R (genes x cells)

# Set row and column names in R
ro.r.assign("counts_matrix", r_counts)
ro.r.assign("cell_metadata_df", r_cell_metadata)
ro.r.assign("gene_metadata_df", r_gene_metadata)
ro.r.assign("gene_names", ro.StrVector(gene_names))
ro.r.assign("cell_names", ro.StrVector(cell_names))

# Create Seurat object in R
ro.r(
    """
# Set dimensions and names
rownames(counts_matrix) <- gene_names
colnames(counts_matrix) <- cell_names
rownames(cell_metadata_df) <- cell_names
rownames(gene_metadata_df) <- gene_names

# Convert to sparse matrix for efficiency
counts_sparse <- Matrix::Matrix(counts_matrix, sparse = TRUE)

# Create Seurat object
cat("Creating Seurat object...\n")
seurat_obj <- CreateSeuratObject(
    counts = counts_sparse,
    meta.data = cell_metadata_df,
    project = "BacSC_3c_filtered"
)

# Add gene metadata to the Seurat object
cat("Adding gene metadata to Seurat object...\n")
seurat_obj[["RNA"]]@meta.data <- gene_metadata_df

# Print basic information
cat("✅ Seurat object created successfully\n")
print(seurat_obj)

# Check gene metadata was added
cat("Gene metadata columns:", colnames(seurat_obj[["RNA"]]@meta.data), "\n")

# Check version
cat("Seurat version:", as.character(packageVersion("Seurat")), "\n")
"""
)

🔄 Creating Seurat object in R...
Creating Seurat object...


R callback write-console: Avis :  
R callback write-console:  Feature names cannot have underscores ('_'), replacing with dashes ('-')
  


Adding gene metadata to Seurat object...
✅ Seurat object created successfully
An object of class Seurat 
5500 features across 2988 samples within 1 assay 
Active assay: RNA (5500 features, 0 variable features)
 1 layer present: counts
Gene metadata columns: gene_id final_gene_name symbol_renamed Gene Type Name Protein accession symbol_old locus_tag_old n_cells_by_counts mean_counts log1p_mean_counts pct_dropout_by_counts total_counts log1p_total_counts n_cells gene_pass_union_OD_5pct gene_pass_intersection_OD_5pct gene_pass_union_OD_10pct gene_pass_intersection_OD_10pct gene_pass_union_OD_15pct gene_pass_intersection_OD_15pct 
Seurat version: 5.3.0 


In [ ]:
# Save as RDS file
print("\n💾 Saving Seurat object as RDS...")

OUTPUT_PATH = "data/RDS"
OUTPUT_FILE = (
    "GeneFull_raw_unique_mult_with_metadata_grouped_filtered_100UMI_3c_SeuratV5.rds"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
output_path = os.path.join(OUTPUT_PATH, OUTPUT_FILE)

# Save in R
ro.r.assign("output_path", output_path)
ro.r(
    """
# Save Seurat object as RDS
cat("Saving to:", output_path, "\n")
saveRDS(seurat_obj, file = output_path)

# Verify file was created
if (file.exists(output_path)) {
    cat("✅ RDS file saved successfully\n")
    cat("📊 File size:", file.size(output_path), "bytes\n")
} else {
    cat("❌ Error: RDS file was not created\n")
}
"""
)


💾 Saving Seurat object as RDS...
Saving to: data/RDS/GeneFull_raw_unique_mult_with_metadata_grouped_filtered_100UMI_3c_SeuratV5.rds 
✅ RDS file saved successfully
📊 File size: 2544372 bytes


In [ ]:
# Verify the saved RDS file
print("\n🔍 Verifying saved RDS file...")

ro.r(
    """
# Test loading the saved RDS file
tryCatch({
    test_obj <- readRDS(output_path)
    cat("✅ RDS file verification successful\n")
    cat("📊 Loaded object dimensions:", dim(test_obj), "\n")
    cat("📊 Object class:", class(test_obj), "\n")
    
    # Show basic info
    cat("\n📋 Basic object information:\n")
    print(test_obj)
    
    # Show available assays
    cat("\n🧬 Available assays:", names(test_obj@assays), "\n")
    
    # Show cell metadata columns
    cat("\n📊 Cell metadata columns:", colnames(test_obj@meta.data), "\n")
    
    # Show gene metadata columns
    cat("📊 Gene metadata columns:", colnames(test_obj[["RNA"]]@meta.data), "\n")
    
}, error = function(e) {
    cat("❌ Error verifying RDS file:", e$message, "\n")
})
"""
)


🔍 Verifying saved RDS file...
✅ RDS file verification successful
📊 Loaded object dimensions: 5500 2988 
📊 Object class: Seurat 

📋 Basic object information:
An object of class Seurat 
5500 features across 2988 samples within 1 assay 
Active assay: RNA (5500 features, 0 variable features)
 1 layer present: counts

🧬 Available assays: RNA 

📊 Cell metadata columns: orig.ident nCount_RNA nFeature_RNA barcode Well_loc_rd1 Well_loc_rd2 Well_loc_rd3 CultureMedium RepBio OD RepTech ODt OD_theoric OD_measured Extracted_Sequence_rd1 Group n_genes RepTech_suffix RepBio_suffix group n_genes_by_counts log1p_n_genes_by_counts total_counts log1p_total_counts pct_counts_in_top_50_genes pct_counts_in_top_100_genes pct_counts_in_top_200_genes pct_counts_in_top_500_genes 
📊 Gene metadata columns: gene_id final_gene_name symbol_renamed Gene Type Name Protein accession symbol_old locus_tag_old n_cells_by_counts mean_counts log1p_mean_counts pct_dropout_by_counts total_counts log1p_total_counts n_cells ge

In [ ]:
# Final summary
print("\n" + "=" * 80)
print("📊 ANNDATA TO SEURAT V5 CONVERSION SUMMARY")
print("=" * 80)

print(f"✅ Input AnnData: {adata_final.n_obs:,} cells × {adata_final.n_vars:,} genes")
print(f"✅ Output RDS file: {output_path}")
print(f"✅ Seurat version: R packages loaded and used")

if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f"✅ File size: {file_size:,} bytes ({file_size/1024/1024:.1f} MB)")
    print("✅ Conversion completed successfully!")
else:
    print("❌ Error: Output file was not created")

print("\n💡 Usage in R:")
print(f"   seurat_obj <- readRDS('{output_path}')")
print("   print(seurat_obj)")
print("   # Available assays: RNA")
print("   # Available cell metadata: all columns from original AnnData obs")
print("   # Available gene metadata: all columns from original AnnData var")
print("   # Access gene metadata: seurat_obj[['RNA']]@meta.data")

print("\n🎉 3d AnnData to Seurat V5 conversion completed!")


📊 ANNDATA TO SEURAT V5 CONVERSION SUMMARY
✅ Input AnnData: 2,988 cells × 5,500 genes
✅ Output RDS file: data/RDS/GeneFull_raw_unique_mult_with_metadata_grouped_filtered_100UMI_3c_SeuratV5.rds
✅ Seurat version: R packages loaded and used
✅ File size: 2,544,372 bytes (2.4 MB)
✅ Conversion completed successfully!

💡 Usage in R:
   seurat_obj <- readRDS('data/RDS/GeneFull_raw_unique_mult_with_metadata_grouped_filtered_100UMI_3c_SeuratV5.rds')
   print(seurat_obj)
   # Available assays: RNA
   # Available cell metadata: all columns from original AnnData obs
   # Available gene metadata: all columns from original AnnData var
   # Access gene metadata: seurat_obj[['RNA']]@meta.data

🎉 3d AnnData to Seurat V5 conversion completed!
